# Final MLR raw versus residualized comparison
Independent CSV-only entry point. Restart Kernel → Run All after configuring paths.
No historical notebook execution, raw GRIB/NetCDF reads, or Stage-II fitting.
All real scientific results require genuine exported HPC inputs. See `README_mlr_comparison.md`.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
# Open this notebook with the repository as the Jupyter working directory,
# or set MLR_CODE_ROOT to the cloned repository path before starting Jupyter.
CODE_ROOT = Path(os.environ.get("MLR_CODE_ROOT", str(Path.cwd()))).expanduser().resolve()
if not (CODE_ROOT / "mlr_comparison_core.py").is_file():
    raise FileNotFoundError(f"Set CODE_ROOT to the repository containing the helper modules: {CODE_ROOT}")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
import mlr_comparison_core as core
INPUT_ROOT = Path(os.environ.get("MLR_INPUT_ROOT", "/data1/huangy/fig6/NC/v12/four_init_full_pipeline_NCVIconcurrent"))
OUTPUT_ROOT = Path(os.environ.get("MLR_OUTPUT_ROOT", str(INPUT_ROOT / "mlr_raw_residual_comparison")))
print("INPUT_ROOT:", INPUT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Initializations:", core.INITS, "Inclusive BJT windows:", core.WINDOWS)
core.output_guard(INPUT_ROOT, OUTPUT_ROOT)

## 2. Input schemas, provenance and historical-file availability

In [ ]:
raw, ncvi, history, provenance = core.read_inputs(INPUT_ROOT)
for record in provenance:
    print(record)
print("Raw schema:", list(raw.columns))
print("NCVI schema:", list(ncvi.columns))

## 3. Final predictor lists and raw counterparts

In [ ]:
mapping = core.predictor_mapping()
print(mapping.to_string(index=False))
predictor_history_status = core.check_predictor_history(history)
print("Historical predictor order:", predictor_history_status)

## 4. Strict paired members and residualization audits

In [ ]:
groups, residualization_audit = core.prepare_pairs(raw, ncvi)
print("Paired groups:", len(groups), "members per group:", [len(g) for g in groups.values()])
print(residualization_audit.to_string(index=False))

## 5. Paired standardized OLS

In [ ]:
models = core.fit_pairs(groups)
print("Full OLS fits:", len(models), "expected: 16")
for key, model in models.items():
    print(key, "R²=", model["R2"], "rank=", model["rank"], "condition=", model["condition_number"])

## 6. Shared MC sequence, Johnson and fixed-column LOFO

In [ ]:
permutations = core.lmg_permutations()
importance_long, model_qc = core.importance(models, permutations)
print(model_qc.to_string(index=False))

## 7. Assemble and save independently auditable numerical tables

In [ ]:
results = core.assemble_results(groups, models, importance_long, model_qc, residualization_audit)
history_qc = core.compare_history(results, history)
manifest = core.save_results(results, history_qc, INPUT_ROOT, OUTPUT_ROOT, provenance, permutations)
print("Saved numerical tables:", list(core.TABLE_FILES.values()))

## 8. Historical reproduction, fitted-space equivalence and QC gate

In [ ]:
print(results["equivalence"].to_string(index=False))
print(history_qc.to_string(index=False))
print("Historical reproduction:", manifest["history_status"])
print("Computational QC:", manifest["computational_status"])
if manifest["computational_status"] != "PASS":
    raise RuntimeError("QC failed/importance undefined; tables were saved for audit. Inspect OUTPUT_ROOT/qc before plotting.")
if manifest["history_status"] != "PASS":
    print("Historical reproduction is incomplete: missing CSVs/fields cannot be certified.")

## 9. Run summary and next notebook

In [ ]:
print(results["cross_init"].to_string(index=False))
print("16 models computed from:", INPUT_ROOT)
print("Tables and QC:", OUTPUT_ROOT)
print("Next: open mlr_comparison_figures.ipynb with the same paths.")
print("These are in-sample associations; no causal conclusion or preferred model version is selected.")